In [14]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

DATA_DIR = '/content/drive/MyDrive/TimeSeriesDeepLearning_FIM601/kaggle_data/optiver-realized-volatility-prediction'

Mounted at /content/drive


In [15]:
trade_train = pd.read_parquet(f"{DATA_DIR}/trade_train.parquet")
book_train = pd.read_parquet(f"{DATA_DIR}/book_train.parquet")
trade_test = pd.read_parquet(f"{DATA_DIR}/trade_test.parquet")
book_test = pd.read_parquet(f"{DATA_DIR}/book_test.parquet")

## Data Breakdown

### Book Data
* `time_id`: The dataset is composed of 10 minute blocks. Each 10 minute block is uniquely identified by its `time_id`. 
* `seconds_in_bucket`: The amount of seconds that has elapsed since the beginning of the bucket identified by `time_id`. This is a snapshot of the market at this particular time. If a particular second is missing, it is assumed that the market is unchanged from the last entry. 
* `[bid/ask]_price[1/2]`: Exactly what it sounds like. `bid_price2` < `bid_price1` < `ask_price1` < `ask_price2`. 
* `stock_id`: Unique stock identifier. 

### Trade Data

In [29]:
trade_train[(trade_train['stock_id'] == 0) & (trade_train['time_id'] == 5)]

,time_id,seconds_in_bucket,price,size,order_count,stock_id
0,5,21,1.002301,326,12,0
1,5,46,1.002778,128,4,0
2,5,50,1.002818,55,1,0
3,5,57,1.003155,121,5,0
4,5,68,1.003646,4,1,0
5,5,78,1.003762,134,5,0
6,5,122,1.004207,102,3,0
7,5,127,1.004577,1,1,0
8,5,144,1.004370,6,1,0
9,5,147,1.003964,233,4,0


In [28]:
book_train[(book_train['stock_id'] == 0) & (book_train['time_id'] == 5)]

,time_id,seconds_in_bucket,bid_price1,ask_price1,bid_price2,ask_price2,bid_size1,ask_size1,bid_size2,ask_size2,stock_id
0,5,0,1.001422,1.002301,1.001370,1.002353,3,226,2,100,0
1,5,1,1.001422,1.002301,1.001370,1.002353,3,100,2,100,0
2,5,5,1.001422,1.002301,1.001370,1.002405,3,100,2,100,0
3,5,6,1.001422,1.002301,1.001370,1.002405,3,126,2,100,0
4,5,7,1.001422,1.002301,1.001370,1.002405,3,126,2,100,0
...,...,...,...,...,...,...,...,...,...,...,...
297,5,585,1.003129,1.003749,1.003025,1.003801,100,3,26,3,0
298,5,586,1.003129,1.003749,1.002612,1.003801,100,3,2,3,0
299,5,587,1.003129,1.003749,1.003025,1.003801,100,3,26,3,0
300,5,588,1.003129,1.003749,1.002612,1.003801,100,3,2,3,0
